In [1]:
import numpy as np
from shapely.geometry import LineString, Point
from shapely.ops import split
import networkx as nx
from shapely.ops import unary_union
from shapely.geometry import LineString, MultiLineString
import pandas as pd
import geopandas as gpd

In [ ]:
df_ocsge = gpd.read_file('/app/data/datasets/debug/bush/OCS-GE_2-0__GPKG_LAMB93_D034_2021-01-01/OCS-GE/1_DONNEES_LIVRAISON_2025-04-00044/OCSGE_2-0_GPKG_LAMB93_D34-2021/OCCUPATION_SOL.gpkg')
df_ocsge_forests = df_ocsge[df_ocsge.code_cs.str.contains('CS2.1.1') | df_ocsge.code_cs.str.contains('CS2.1.2')]
df_ocsge_forests

In [ ]:
df_ocsge_forests_exploded = df_ocsge_forests.explode(index_parts=False).reset_index(drop=True)
df_ocsge_forests_exploded

In [ ]:


def shared_border_length_buffered(g1, g2, tol=5.0):
    """
    Length of the longest contiguous shared border between two polygons,
    using a buffer tolerance to detect contact.
    """

    # Detect adjacency with tolerance
    inter = g1.boundary.buffer(tol).intersection(g2.boundary)

    #print(f"intersection is : {inter}")

    if inter.is_empty:
        return 0.0

    # Project intersection back to original boundary
    # projected = inter.intersection(g1.boundary)
    projected = inter

    if projected.is_empty:
        return 0.0

    if isinstance(projected, LineString):
        return projected.length

    if isinstance(projected, MultiLineString):
        return max(line.length for line in projected.geoms)

    return 0.0

BUFFER_TOL = 5.0      # meters (robustness)
MIN_BORDER_LENGTH = 50.0  # meters

sindex = df_ocsge_forests_exploded.sindex
geoms = df_ocsge_forests_exploded.geometry

G = nx.Graph()
G.add_nodes_from(geoms.index)

for i, geom in geoms.items():
    # spatial index pre-filter
    for j in sindex.intersection(geom.bounds):
        
        #print(f"processing intesections between {i} and {j}")
        if j <= i:
            continue

        length = shared_border_length_buffered(geom, geoms[j],tol=BUFFER_TOL)
        #print(f"border detected between {i} and {j}, length : {length}")
        if length > geom.boundary.length * 0.25 or length > geoms[j].boundary.length * 0.25 or length > MIN_BORDER_LENGTH :
            G.add_edge(i, j)
                
merged_rows = []

for component in nx.connected_components(G):
    subset = df_ocsge_forests_exploded.loc[list(component)]
    merged_geom = unary_union(subset.geometry)
    
    merged_rows.append({
        "geometry": merged_geom
    })

gdf_polygons = gpd.GeoDataFrame(
    merged_rows,
    geometry="geometry",
    crs=df_ocsge_forests_exploded.crs
)

gdf_polygons

In [ ]:
gdf_polygons.to_file('/app/data/datasets/debug/bush/ocsge_forests_exploded.gpkg',driver='GPKG')

In [ ]:


def barycentric_ray(boundary_point, centroid, scale=1000):
    """
    Create a long line passing through boundary_point -> centroid -> opposite side
    """
    v = np.array(centroid.coords[0]) - np.array(boundary_point.coords[0])
    v = v / np.linalg.norm(v)

    p1 = boundary_point.coords[0]
    p2 = centroid.coords[0] + v * scale

    return LineString([p1, p2])

def my_distance(polygon, boundary_point):
    centroid = polygon.centroid
    ray = barycentric_ray(boundary_point, centroid)

    inter = polygon.intersection(ray)

    if inter.is_empty:
        return None

    if inter.geom_type == "MultiLineString":
        # keep the longest segment inside polygon
        return max(seg.length for seg in inter.geoms)

    return inter.length

def angular_boundary_points(polygon, n=10):
    centroid = polygon.centroid
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)

    points = []
    for a in angles:
        direction = np.array([np.cos(a), np.sin(a)])
        far_point = Point(
            centroid.x + direction[0]*1e6,
            centroid.y + direction[1]*1e6
        )
        ray = LineString([centroid, far_point])
        inter = polygon.boundary.intersection(ray)

        if inter.is_empty:
            continue

        if inter.geom_type == "MultiPoint":
            # take closest intersection to centroid
            p = min(inter.geoms, key=lambda g: g.distance(centroid))
        else:
            p = inter

        points.append(p)

    return points

def my_distances_angular(polygon, n=10):
    points = angular_boundary_points(polygon, n)
    distances = [my_distance(polygon, p) for p in points]
    return [d for d in distances if d is not None]

In [ ]:
def my_distance_features(polygon, n=10, method="angular"):
    dists = my_distances_angular(polygon, n)

    if not dists:
        return {
            "myDist_min": 0,
            "myDist_mean": 0,
            "myDist_std": 0
        }

    return {
        "myDist_min": np.min(dists),
        "myDist_mean": np.mean(dists),
        "myDist_std": np.std(dists)
    }

In [ ]:
features = gdf_polygons.geometry.apply(
    lambda g: pd.Series(my_distance_features(g, n=10, method="angular"))
)

gdf = pd.concat([gdf_polygons, features], axis=1)

In [ ]:
gdf['forest_type'] = 0
gdf.loc[(gdf.myDist_mean > 200) & (gdf.myDist_min > 50),'forest_type'] = 1
gdf

In [ ]:
gdf.to_file('/app/data/datasets/debug/bush/ocsge_forests_clean_types_v3.gpkg',driver='GPKG')